In [ ]:
# 0. 패키지 버전 맞추기 — Colab에서 처음 한 번: 이 셀 실행 → 런타임 다시 시작 → 다음 셀부터
# (google-colab·numba·cudf 버전 경고와 bottleneck 경고는 무시해도 된다)
# 로컬은 .venv에 requirements.txt로 이미 설치되어 있으면 건너뛴다: python -m pip install -r requirements.txt
import sys
if 'google.colab' in sys.modules:
    !pip install -q pandas==3.1.0 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 xgboost==3.4.1 lightgbm==4.7.0 pyarrow==26.0.0 PyYAML==6.0.3
    print('설치 끝: 런타임 → 세션 다시 시작 후 다음 셀부터 실행')
else:
    print('로컬: 설치 생략 (requirements.txt 버전 사용)')

# 튜닝: 트리 수 500 고정 vs 해마다 선택 (부스팅 1등 설정)

LightGBM 순위 회귀 깊이 3에서 트리 수를 해마다 고르는 대신 300~900개 어디로 고정해도 +0.004~0.005였다(`tuning_fixed_lambda`, 2001 기준). 같은 일이 다른 부스팅 모델에서도 일어나는지 본다. 고정값은 구간 가운데 값 **500** 하나로 정해 두고 고르지 않는다(1등 값 410을 쓰면 검증으로 고른 부풀림이 생김).

| 방식 | H3 기간 | 트리 수 |
| --- | --- | --- |
| 기준 | 해마다(직전 1년 내부 검증) | 해마다 1~1000 |
| ① | 해마다: 트리 500 지점의 내부 검증 IC가 가장 높은 기간(같으면 긴 기간) | 500 |
| ② | 1년·3년·5년 각각 고정 | 500 |

- Fold마다 H3 3개 × 1000트리를 한 번 학습하고(공용 `select_and_fit_boosting`, 기본 실행과 같은 학습), 세 방식은 예측만 다르다. 그래서 시간은 기본 실행의 그 설정 하나와 같다.
- 기준 줄은 기본 실행 결과(`performance_summary.csv`의 같은 설정, 전체 종목)와 같아야 한다.
- 판정(2026-11-17 결정): ①이나 ②가 기준보다 0.002 넘게 높고 여러 모델에서 같은 방향일 때 바꾼다. ②에서 한 기간만 높으면 그 기간을 고른 것 자체가 부풀림일 수 있다.
- 결과: `model_results/<AUTHOR>/tuning_fixed_trees_<colab|local>.csv`(설정 × 방식 한 줄), `..._folds.csv`(Fold별 고른 값). 설정 하나가 끝날 때마다 저장한다.

## 0. 실행 환경 확인

In [ ]:
import os, sys, platform, subprocess, importlib.util, shutil
from pathlib import Path

print('python      :', sys.version.split()[0], platform.platform())
print('작업 폴더    :', os.getcwd())
print('/content 존재:', Path('/content').exists())

# 저장소가 보이는지: 위로 올라가며 src/와 config/experiment.yaml을 찾는다
found = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file()), None)
print('저장소 루트   :', found)

if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], capture_output=True, text=True).stdout)
else:
    print('GPU          : 없음')

for m in ['numpy', 'pandas', 'pyarrow', 'scipy', 'sklearn', 'xgboost', 'lightgbm', 'yaml']:
    if importlib.util.find_spec(m):
        mod = __import__(m)
        print(f'{m:9s}', getattr(mod, '__version__', '?'))
    else:
        print(f'{m:9s}', '설치 안 됨')

## 1. 경로·설정

- `SETTINGS`에 비교할 설정을 적는다. XGBoost는 Colab(GPU), LightGBM은 로컬(CPU).
- Colab은 `GIT_REF`에 브랜치 이름(`jiun`), 결과는 내 드라이브 `model_results/<AUTHOR>/`. 내려받아 로컬 `outputs/model_results/<AUTHOR>/`에 넣는다.

In [ ]:
import os, sys, re, subprocess
from datetime import datetime
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/JiunShin1214/finance_analytics.git'
GIT_REF = ''              # Colab에서만: 브랜치 이름(예: 'main', 'hong') — 그 브랜치의 GitHub 최신 코드. 특정 시점 재현은 커밋 해시
AUTHOR = ''              # 결과 폴더 outputs/model_results/<작성자>/ (영문·숫자·_·-만)

# 비교할 설정: (종류, 목표값, 손실, 깊이, 학습률). 모델 종류별 기본 실행 1등 설정을 적는다.
# XGBoost는 Colab GPU, LightGBM은 로컬 CPU에서 돌린다(고를 때 장치 = 최종 장치).
SETTINGS = [
    # ('xgb_rankreg', 'percentile', 'mse', 3, 0.01),
    # ('xgb_pairwise', 'decile', 'pairwise', 3, 0.01),
    # ('xgb_reg', 'monthly_zscore', 'huber_sigma135', 1, 0.01),
    # ('lgb_reg', 'monthly_zscore', 'huber_sigma135', 3, 0.01),
]
TREES_FIXED = 500        # 2026-11-17 결정: LightGBM 고정 결과 300~900 구간 가운데 값
DEVICE = None            # None = config(XGBoost cuda, LightGBM cpu). 시험할 때만 'cpu'
FOLD_ONLY = None         # None = 16개 Fold 전부. 시험할 때만 Fold 번호 목록(예: [1])
if not SETTINGS:
    raise ValueError('SETTINGS에 비교할 설정을 적는다.')
if not AUTHOR:
    raise ValueError('AUTHOR를 채운다(결과 폴더 outputs/model_results/<AUTHOR>/).')
DRIVE_BASE = Path('/content/drive/MyDrive/금융데이터인텔리전스')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not GIT_REF:
        raise RuntimeError("GIT_REF에 브랜치 이름(예: 'main', 'hong')을 적는다.")
    ROOT = Path('/content/finance_analytics')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
    # GitHub에서 최신을 받아 GIT_REF로 맞춘다: 브랜치 이름이면 원격 최신(origin/<브랜치>), 아니면 커밋 해시
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', '--prune'], check=True)
    _remote = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--verify', '--quiet', f'origin/{GIT_REF}'],
                             capture_output=True, text=True).stdout.strip()
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', _remote or GIT_REF], check=True)
    print('받은 코드:', GIT_REF, '@', subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--short', 'HEAD'],
                                               capture_output=True, text=True, check=True).stdout.strip())
    if not (ROOT / 'data').is_symlink():       # 저장소의 data/(.gitkeep만 있음)를 Drive 공유 data로 바꿔 연결
        (ROOT / 'data').rename(ROOT / 'data_repo_placeholder')
        (ROOT / 'data').symlink_to(DRIVE_BASE / 'data')
    OUT_ROOT = Path('/content/drive/MyDrive/model_results')   # Colab: 내 드라이브(개인)에 같은 구조로 → 내려받아 로컬 outputs/model_results/에 넣고 커밋
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file())
    OUT_ROOT = None    # 기본: ROOT/outputs/model_results (<작성자>/<결과 폴더>)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
import pandas as pd
from src.evaluation import model_test as mt

config = yaml.safe_load((ROOT / 'config' / 'experiment.yaml').read_text(encoding='utf-8'))

# requirements.txt 고정 버전과 설치 버전 비교(다르면 경고)
pins = dict(re.findall(r'^([A-Za-z0-9_.-]+)==([^\s#]+)', (ROOT / 'requirements.txt').read_text(encoding='utf-8'), re.M))
for name in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'pyarrow', 'PyYAML', 'xgboost', 'lightgbm']:
    try:
        got = version(name)
    except PackageNotFoundError:
        got = '없음'
    print(f"{name:13s} 고정 {pins.get(name, '?'):8s} 설치 {got:8s} {'' if got == pins.get(name) else '← 다름'}")

started = datetime.now()
pd.set_option('display.width', 200, 'display.max_rows', 200, 'display.float_format', '{:.4f}'.format)
print('ROOT   :', ROOT)
import shutil
from src.models import boosting
GPU_NAME = (subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
            if shutil.which('nvidia-smi') else None)
USE_GPU = DEVICE is None and any(boosting.default_device(s[0]) == 'cuda' for s in SETTINGS)
if USE_GPU and not GPU_NAME:
    raise RuntimeError('XGBoost는 config상 GPU(cuda)로 고르고 최종 모델도 GPU다. GPU 런타임으로 바꿔 실행한다.')
N_THREADS = os.cpu_count() if IN_COLAB else 6
OUT_DIR = (OUT_ROOT or ROOT / config['PATHS']['model_results']) / AUTHOR
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLACE = 'colab' if IN_COLAB else 'local'
OUT_CSV = OUT_DIR / f'tuning_fixed_trees_{PLACE}.csv'
OUT_FOLDS = OUT_DIR / f'tuning_fixed_trees_{PLACE}_folds.csv'
print('저장   :', OUT_CSV)
print('GPU    :', GPU_NAME, '| 스레드:', N_THREADS, '| 시작:', started.isoformat(timespec='seconds'))

## 2. 입력 준비 (약 2분)

부스팅은 순위 입력. 학습 시작 2001년, 검증 예측 133,876행을 확인한다.

In [ ]:
import time
import numpy as np
t0 = time.perf_counter()
prepared = mt.prepare_inputs(ROOT, config)
print(f'입력 준비 {time.perf_counter() - t0:.0f}초 | 학습 시작 {prepared["checks"]["train_start_target_year"]}'
      f' | 검증 예측 행 {prepared["checks"]["validation_prediction_rows"]:,}')
data = prepared['data']
labels = mt.boosting_labels(prepared)
decay_of = mt._decay_of(prepared)
val = mt.validation_rows(prepared, config)
FOLDS = [f for f in mt.plan_folds(config['MODEL_GRID']) if FOLD_ONLY is None or f['fold'] in FOLD_ONLY]
INPUTS = ['size', 'bm', 'mom', 'sigma']
YEARS = {'span12': '1년', 'span36': '3년', 'span60': '5년'}
print('H3 후보:', labels, '| Fold', len(FOLDS))

## 3. 실행 (설정 하나당 기본 실행과 같은 시간)

대략: XGBoost 순위 회귀 깊이 3 약 6분, 회귀 약 6분, pairwise 약 25분(GPU), LightGBM 회귀 약 7분(CPU 6스레드) — 기본 실행 시간에서 나눈 값.

In [ ]:
def monthly(pred):
    t = mt.monthly_ic(val.target_month.to_numpy(), pred, val.y.to_numpy(float))
    return t.ic.where(t.reason.ne('constant_score'), 0.0)


def at_fixed(x):
    v = float(x['curve'][TREES_FIXED - 1])
    return v if np.isfinite(v) else -np.inf


METHODS = ['기준(해마다)', '① H3 해마다 + 트리 500'] + [f'② H3 {YEARS[l]} + 트리 500' for l in labels]
ROWS, FOLD_ROWS, MONTHLY = [], [], {}
for setting in SETTINGS:
    kind = setting[0]
    device = DEVICE or boosting.default_device(kind)
    name = f'{kind} {setting[1]} {setting[2]} d{setting[3]}'
    preds = {m: np.full(len(val), np.nan) for m in METHODS}
    t0 = time.perf_counter()
    for f in FOLDS:
        m = mt.fold_rows(prepared, f)
        ctx = mt.FoldContext(data, m['fit'], m['inner'], m['out'])
        out = val.fold.eq(f['fold']).to_numpy()
        chosen, _, fits = mt.select_and_fit_boosting(ctx, setting, input_cols=INPUTS, sigma_labels=labels, decay_of=decay_of,
                                                     device=device, n_threads=N_THREADS)
        pred = lambda x, k: boosting.predict_boosting(x['fitted'], ctx.X(x['columns'], 'out'), k)
        preds['기준(해마다)'][out] = pred(chosen, chosen['trees'])
        best = max(fits, key=lambda x: (at_fixed(x), x['decay']))       # 트리 500 지점 내부 IC 최고, 같으면 긴 기간
        preds['① H3 해마다 + 트리 500'][out] = pred(best, TREES_FIXED)
        for x in fits:
            preds[f"② H3 {YEARS[x['h3_label']]} + 트리 500"][out] = pred(x, TREES_FIXED)
        FOLD_ROWS.append({'setting': name, 'fold': f['fold'], 'valid_year': f['valid_year'],
                          'base_h3': chosen['h3_label'], 'base_trees': chosen['trees'], 'base_inner_ic': chosen['inner_ic'],
                          'fixed_h3': best['h3_label'], 'fixed_inner_ic': at_fixed(best),
                          **{f"inner_ic_500_{x['h3_label']}": at_fixed(x) for x in fits}})
        print(f"  {name} Fold {f['fold']:>2} ({f['valid_year']}): 기준 {chosen['h3_label']}·트리 {chosen['trees']}, "
              f"① {best['h3_label']}, {time.perf_counter() - t0:.0f}초")
    base = monthly(preds['기준(해마다)'])
    for meth in METHODS:
        ic = monthly(preds[meth])
        MONTHLY[(name, meth)] = ic
        ROWS.append({'setting': name, 'method': meth, 'trees_fixed': TREES_FIXED, 'months': int(ic.notna().sum()),
                     'mean_ic': ic.mean(), 'icir': ic.mean() / ic.std(), 'ic_pos': (ic > 0).mean(),
                     'diff_vs_base': ic.mean() - base.mean(), 'device': device,
                     'minutes': round((time.perf_counter() - t0) / 60, 1)})
    pd.DataFrame(ROWS).to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
    pd.DataFrame(FOLD_ROWS).to_csv(OUT_FOLDS, index=False, encoding='utf-8-sig')
    print(f'{name}: 저장 {OUT_CSV.name}, {(time.perf_counter() - t0) / 60:.1f}분')

## 4. 비교

기준과의 차이가 0.002를 넘으면 '검토'. 기준 줄은 기본 실행의 같은 설정 평균 IC(전체 종목)와 같아야 한다.

In [ ]:
res = pd.DataFrame(ROWS)
res['판단'] = np.where(res.method.eq('기준(해마다)'), '기준', np.where(res.diff_vs_base > 0.002, '검토', '기준 유지'))
display(res[['setting', 'method', 'mean_ic', 'diff_vs_base', 'icir', 'ic_pos', 'minutes', '판단']])
fr = pd.DataFrame(FOLD_ROWS)
print('해마다 고른 트리 수 범위:')
display(fr.groupby('setting').base_trees.describe()[['min', '50%', 'max']])